<a href="https://colab.research.google.com/github/TatekBelay/GEARS/blob/master/NDVI_and_KNDVI_Workflow_for_Taiwan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##NDVI and KNDVI Workflow for Taiwan (NOAA CDR, Shapefile AOI)

Where your methodology text conflicts with itself.
• Sensor. The NOAA CDR NDVI product (0.05°, daily) is AVHRR-based, but the text then cites MOD13A1 quality flags (MODIS, 500 m, 16-day) and MODIS GPP. MOD13A1 flags cannot be applied to CDR pixels. The code uses the CDR product's own QA layer.
• Period. You state 1985–2025; I believe the CDR record starts earlier (1981), so the code simply selects 1985–2025. Confirm the end year is actually available.
• Equation 1. It is a plain additive model (KNDVI = Trend + Season + Residual) and is implemented as such; "Residual" is capitalised inconsistently, which is cosmetic.
• Out of scope. TerraClimate drivers, GPP, land cover and the compound drought–heat event definition are not part of this request and are not implemented. No threshold for events was given, so none is invented

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import shapely
from statsmodels.tsa.seasonal import STL

In [ ]:
# ---------------- Configuration (edit here only) ----------------
NDVI_ROOT  = Path("data/ncei_ndvi")   # local download folder, filled in Step 1b
AOI_SHP    = Path("Taiwan_New/Selected/Taiwan_official_new.shp")
OUT_DIR    = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
START_YEAR, END_YEAR = 1985, 2025          # period stated in the methodology
SPRING = (3, 4, 5)                         # March–May
SUMMER = (6, 7, 8)                         # June–August
NDVI_VAR, QA_VAR = "NDVI", "QA"           # VERIFY against the product (see last section)
SCALE_FACTOR_FALLBACK = 0.0001             # used only if the file lacks scale_factor
QA_BAD_BITS = {"AVHRR": (1, 2, 3)}                           # 1 cloudy, 2 cloud shadow, 3 water; no VIIRS entry until its QA layout is checked
STL_PERIOD = 12                            # monthly series, annual cycle

In [ ]:
# ---------------- Step 1: AOI ----------------
aoi = gpd.read_file(AOI_SHP)
assert len(aoi) > 0, "AOI shapefile is empty"
print("Native CRS:", aoi.crs)               # expected: GCS_TWD_1997
aoi = aoi[~aoi.geometry.is_empty].copy()
aoi["geometry"] = aoi.geometry.make_valid()
aoi = aoi.to_crs("EPSG:4326")               # match the NOAA lat/lon grid
print(aoi.drop(columns="geometry").head(19))
minx, miny, maxx, maxy = aoi.total_bounds   # used to subset the global grid early
print("AOI bounds (lon/lat):", minx, miny, maxx, maxy)

###Step 1b: Downloading the 1985–2025 daily files from NCEI
Explanation. The directory https://www.ncei.noaa.gov/data/land-normalized-difference-vegetation-index/access/ is the download point listed on the NCEI product page. NCEI describes the record as running from 1981 to about ten days before the present, built from AVHRR and, later, VIIRS. Two consequences matter for a 1985–2025 study.
• Sensor change. The file names I could confirm (for example AVHRR-Land_v005_AVH13C1_NOAA-09_19880426_c20170613192416.nc, stored in a per-year folder) are AVHRR names. I could not see the VIIRS-era names, so the script selects files by the 8-digit date in the name, and Step 3 stops on VIIRS files until you confirm their QA layout.
• Unverified index page. The directory page was closed to my own fetch tool (robots restriction), so I could not inspect it. The script assumes a standard HTML index of links (1985/, 1986/, … and *.nc inside each) and fails loudly if it receives anything else. NCEI also lists an Amazon S3 mirror for this product, which I have not checked.
The download has four parts: list the files; preview the size before committing disk space; download with retries, a size check against the server's Content-Length, and skipping of finished files (an interrupted run can simply be restarted); then write a manifest and report missing dates. Four parallel connections are used, to stay polite to the server. Install requests first.

In [ ]:
import csv, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from html.parser import HTMLParser
from urllib.parse import urljoin
import requests
BASE_URL  = "https://www.ncei.noaa.gov/data/land-normalized-difference-vegetation-index/access/"
N_WORKERS, TIMEOUT, MAX_RETRY = 4, 60, 5
_tls = threading.local()

def http() -> requests.Session:
    """One Session per thread (requests.Session is not documented as thread-safe)."""
    if not hasattr(_tls, "s"):
        _tls.s = requests.Session()
        _tls.s.headers["User-Agent"] = "ndvi-research-download/1.0"
    return _tls.s

class _Links(HTMLParser):
    def __init__(self):
        super().__init__(); self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            self.links += [v for k, v in attrs if k == "href" and v]

def links_of(url: str) -> list:
    r = http().get(url, timeout=TIMEOUT); r.raise_for_status()
    p = _Links(); p.feed(r.text); return p.links

# ---- (1) List remote files for START_YEAR..END_YEAR -------------------------
year_dirs = {int(l.strip("/")): urljoin(BASE_URL, l)
             for l in links_of(BASE_URL) if re.fullmatch(r"\d{4}/?", l)}
missing_years = [y for y in range(START_YEAR, END_YEAR + 1) if y not in year_dirs]
assert not missing_years, f"Year folders not found on server: {missing_years}"

remote = {}                                   # file name -> (url, date)
for y in range(START_YEAR, END_YEAR + 1):
    for l in links_of(year_dirs[y]):
        m = re.search(r"_(\d{8})_", l)
        if l.endswith(".nc") and m and pd.Timestamp(m.group(1)).year == y:
            remote[Path(l).name] = (urljoin(year_dirs[y], l), pd.Timestamp(m.group(1)))
print(f"{len(remote)} files listed (calendar days in period: {(pd.Timestamp(END_YEAR,12,31)-pd.Timestamp(START_YEAR,1,1)).days+1})")
assert remote, "No .nc files parsed: inspect the index page and adjust the parser"

# ---- (2) Preview the size before downloading --------------------------------
sample = list(remote.values())[:: max(1, len(remote) // 8)][:8]
sizes = [int(http().head(u, timeout=TIMEOUT, allow_redirects=True).headers.get("Content-Length", 0)) for u, _ in sample]
print(f"Mean size of {len(sizes)} sampled files: {np.mean(sizes)/1e6:.1f} MB; "
      f"estimated total: {np.mean(sizes)*len(remote)/1e9:.0f} GB (rough: sampled, not measured in full)")

# ---- (3) Download with retry, size check, skip-if-complete ------------------
def download_one(name: str) -> tuple:
    url, d = remote[name]
    dest = NDVI_ROOT / str(d.year) / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, MAX_RETRY + 1):
        try:
            head = http().head(url, timeout=TIMEOUT, allow_redirects=True)
            expected = int(head.headers.get("Content-Length", -1))
            if dest.exists() and expected in (-1, dest.stat().st_size):
                return name, "skipped", dest.stat().st_size
            part = dest.with_name(name + ".part")
            with http().get(url, stream=True, timeout=TIMEOUT) as r:
                r.raise_for_status()
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            if expected != -1 and part.stat().st_size != expected:
                raise IOError(f"size mismatch {part.stat().st_size} != {expected}")
            part.replace(dest)
            return name, "downloaded", dest.stat().st_size
        except (requests.RequestException, IOError):
            time.sleep(2 ** attempt)
    return name, "FAILED", 0

results = []
with ThreadPoolExecutor(N_WORKERS) as ex:
    futs = [ex.submit(download_one, n) for n in remote]
    for i, f in enumerate(as_completed(futs), 1):
        results.append(f.result())
        if i % 500 == 0:
            print(f"{i}/{len(futs)} processed")

# ---- (4) Manifest and completeness report ----------------------------------
with open(OUT_DIR / "download_manifest.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["file", "status", "bytes", "url", "checked_utc"])
    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for n, s, b in sorted(results):
        w.writerow([n, s, b, remote[n][0], now])
failed = [n for n, s, _ in results if s == "FAILED"]
print("Failed:", len(failed), failed[:10])

all_days   = pd.date_range(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31")
have_days  = pd.DatetimeIndex(sorted(d for _, d in remote.values()))
missing    = all_days.difference(have_days)
print(f"Days with no file on the server: {len(missing)}")
print(pd.Series(missing.to_period('M')).value_counts().sort_index())  # gaps by month

Inputs: the NCEI directory and NDVI_ROOT (the local destination set in Step 1). Outputs: the files in NDVI_ROOT/<year>/, outputs/download_manifest.csv for your methods record, and a list of missing days.
Two points to act on after this step. (a) If a whole calendar month has no data, the strict gap rule in Step 7 (MAX_MISSING_FRAC = 0.0) will turn every pixel to NaN; decide on a gap-handling rule at that point. (b) I could not measure file sizes from here, so rely on the printed size estimate before starting; global daily files may be large

Step 2: Locating and reading the NOAA CDR daily files
Explanation. The product is one NetCDF file per day on a 0.05° global grid. The code walks the access directory, extracts the acquisition date from each file name, and keeps the 1985–2025 files. The date pattern _YYYYMMDD_cYYYYMMDD and the one-folder-per-year layout match AVHRR file names seen on NCEI THREDDS pages; VIIRS-era names could not be checked; the parser skips any file that does not match, and the printed count lets you detect a wrong pattern immediately. Each file is cropped to the AOI bounding box (plus one pixel of margin) on opening, so memory use stays small. Coordinate names are normalised to lat/lon because I cannot confirm the exact names from here.

In [ ]:
def list_daily_files(root: Path, y0: int, y1: int) -> dict:
    """Map date -> file path for every file whose date lies in [y0, y1]."""
    out = {}
    for p in sorted(root.rglob("*.nc")):
        m = re.search(r"_(\d{8})_", p.name)
        if m is None:
            continue                              # not a daily CDR granule
        d = pd.Timestamp(m.group(1))
        if y0 <= d.year <= y1:
            out[d] = p
    return out

daily_files = list_daily_files(NDVI_ROOT, START_YEAR, END_YEAR)
print(f"{len(daily_files)} daily files found; expected about 14,975 for 1985-2025")
assert daily_files, "No files matched: check NDVI_ROOT and the file-name pattern"

# Inspect one file once, to confirm variable names, scale_factor and QA attributes
first = next(iter(daily_files.values()))
with xr.open_dataset(first) as ds0:
    print(ds0)
    print(ds0[NDVI_VAR].attrs)
    print(ds0[QA_VAR].attrs)

def open_subset(path: Path) -> xr.Dataset:
    """Open one daily file, standardise coordinate names, crop to the AOI box."""
    ds = xr.open_dataset(path, mask_and_scale=True)   # applies scale_factor, _FillValue -> NaN
    ren = {}
    for old, new in (("latitude", "lat"), ("longitude", "lon")):
        if old in ds.coords and new not in ds.coords:
            ren[old] = new
    ds = ds.rename(ren).sortby("lat").sortby("lon")
    pad = 0.05                                        # one pixel of margin
    return ds.sel(lat=slice(miny - pad, maxy + pad), lon=slice(minx - pad, maxx + pad))

##Step 3: NDVI and quality-flag masking
Explanation. NDVI is the normalised ratio of near-infrared (NIR) and red reflectance (Rouse et al., 1974; Tucker, 1979):

\mathrm{NDVI} = \frac{\rho_{NIR} - \rho_{Red}}{\rho_{NIR} + \rho_{Red}}

The NOAA CDR product already supplies NDVI as a variable, so it is read and decoded (scale factor and fill value) rather than recomputed from bands; ndvi_from_bands documents the equation only. Quality masking uses the QA bit definitions published in the Earth Engine catalogue entry for this product (a secondary source; please confirm them in the NOAA User Guide): bit 1 cloudy, bit 2 cloud shadow, bit 3 over water, bit 4 sunglint, bit 6 night, bits 8–14 invalid channel or BRDF flags. The code rejects bits 1, 2 and 3, the cloud and water categories named in your methodology; bits 4, 6 and 8–14 are optional additions you can append. Bit 5 ("dense dark vegetation") and bit 7 ("channels 1–5 are valid") are not rejection flags and must not be masked. This QA layer has no snow bit, so the snow exclusion in your methodology cannot be applied through QA. The definitions are for the AVHRR files; I could not confirm that VIIRS-era files (2014 onward) use the same layout, so QA_BAD_BITS has no VIIRS entry and the function stops on a VIIRS file until you add one.

In [ ]:
def ndvi_from_bands(nir, red):
    """Reference equation only (not used: the CDR provides NDVI directly)."""
    return (nir - red) / (nir + red)

def sensor_of(path: Path):
    for s in ("AVHRR", "VIIRS"):
        if path.name.startswith(s):
            return s
    return None

def daily_ndvi(path: Path) -> xr.DataArray:
    """Return QA-masked NDVI for one day, dims (lat, lon), plus a time coordinate."""
    sensor = sensor_of(path)
    if sensor not in QA_BAD_BITS:
        raise ValueError(f"No verified QA bit list for {path.name} (sensor={sensor}); "
                         "add it to QA_BAD_BITS after checking the User Guide.")
    ds = open_subset(path)
    ndvi = ds[NDVI_VAR]
    qa   = ds[QA_VAR]
    if "time" in ndvi.dims:
        ndvi = ndvi.squeeze("time", drop=True)
        qa   = qa.squeeze("time", drop=True)

    # Decode QA as an integer bit field; a missing QA value is treated as invalid
    bad_mask = sum(1 << b for b in QA_BAD_BITS[sensor])
    qa_int   = qa.fillna(0).astype("int64")
    bad      = qa.isnull() | ((qa_int & bad_mask) != 0)

    ndvi = ndvi.where(~bad)
    ndvi = ndvi.where((ndvi >= -1.0) & (ndvi <= 1.0))   # physical range of NDVI
    ndvi.attrs.clear()
    return ndvi.astype("float32")

Step 4: KNDVI
Explanation. The kernel NDVI replaces the linear NIR–red difference with a kernel similarity (Camps-Valls et al., 2021). With the radial basis function kernel and the length-scale set to σ = (NIR + Red)/2, it reduces to the expression given in your methodology:
\mathrm{kNDVI} = \tanh\!\left(\mathrm{NDVI}^{2}\right)
The function is monotonic for NDVI ≥ 0 and compresses high values, which is how it reduces saturation. One property matters for your data: because NDVI is squared, a negative NDVI (water, snow) would map to a positive kNDVI. Your methodology removes water and snow pixels through the QA step, so no extra rule is applied; the code only reports how many negative values survive, so you can decide.

In [ ]:
def kndvi(ndvi: xr.DataArray) -> xr.DataArray:
    """kNDVI = tanh(NDVI^2) (Camps-Valls et al., 2021, with sigma = (N+R)/2)."""
    k = xr.apply_ufunc(np.tanh, ndvi ** 2)
    k.name = "kndvi"
    return k

Step 5: Monthly Maximum Value Composite and seasonal means

Explanation. Following the methodology, daily kNDVI is composited to monthly values with the Maximum Value Composite (MVC; Holben, 1986): for each pixel and month the largest valid daily value is kept, which suppresses residual cloud and atmospheric contamination. Monthly values are then averaged over spring (March–May) and summer (June–August) for each year. Months with no valid day stay NaN; the seasonal mean uses whichever of the three months are valid, because the methodology gives no minimum-month rule (add one here if you want it). Since tanh(x²) is monotonic for x ≥ 0, compositing kNDVI gives the same result as computing kNDVI from the composited NDVI for non-negative NDVI.

In [ ]:
# Build the daily stack (about 15,000 small 2-D arrays; this is the slow, I/O-bound step)
ndvi_daily = xr.concat(
    [daily_ndvi(p).expand_dims(time=[d]) for d, p in daily_files.items()],
    dim="time", join="exact",          # fails loudly if grids differ between files
)
kndvi_daily = kndvi(ndvi_daily)

n_neg = int((ndvi_daily < 0).sum())
print(f"Negative NDVI values surviving QA masking: {n_neg} (they map to positive kNDVI)")

# Monthly MVC: maximum valid daily value per calendar month
kndvi_monthly = kndvi_daily.resample(time="1MS").max(skipna=True)
ndvi_monthly  = ndvi_daily.resample(time="1MS").max(skipna=True)

def seasonal_mean(monthly: xr.DataArray, months: tuple) -> xr.DataArray:
    """Mean of the monthly values over the given calendar months, per year."""
    sel = monthly.sel(time=monthly.time.dt.month.isin(months))
    return sel.groupby("time.year").mean("time", skipna=True)

kndvi_spring = seasonal_mean(kndvi_monthly, SPRING)
kndvi_summer = seasonal_mean(kndvi_monthly, SUMMER)
ndvi_spring  = seasonal_mean(ndvi_monthly,  SPRING)
ndvi_summer  = seasonal_mean(ndvi_monthly,  SUMMER)

Output: monthly and seasonal NDVI and kNDVI arrays with dims (time or year, lat, lon).

Explanation. A pixel belongs to a polygon when its centre lies inside it (shapely.contains_xy). One mask is built for the whole AOI and one per COUNTY_ID. With 0.05° pixels, narrow coastal strips and the smallest island polygons contain few or no pixel centres, so their pixel counts are printed; treat counties with very few pixels with caution. The pixel-centre rule is a design choice, not something your methodology specifies

In [ ]:
import shapely

def build_masks(template: xr.DataArray, aoi: gpd.GeoDataFrame):
    lon2d, lat2d = np.meshgrid(template.lon.values, template.lat.values)
    coords = {"lat": template.lat, "lon": template.lon}
    whole = xr.DataArray(
        shapely.contains_xy(shapely.union_all(aoi.geometry.values), lon2d, lat2d),
        coords=coords, dims=("lat", "lon"))
    per_county = {
        str(cid): xr.DataArray(shapely.contains_xy(geom, lon2d, lat2d),
                               coords=coords, dims=("lat", "lon"))
        for cid, geom in zip(aoi["COUNTY_ID"], aoi.geometry)
    }
    return whole, per_county

aoi_mask, county_masks = build_masks(kndvi_monthly.isel(time=0), aoi)
print("Pixels in AOI:", int(aoi_mask.sum()))
print({k: int(v.sum()) for k, v in county_masks.items()})

# Clip every product to the AOI (pixels outside become NaN)
kndvi_monthly = kndvi_monthly.where(aoi_mask)
kndvi_spring  = kndvi_spring.where(aoi_mask)
kndvi_summer  = kndvi_summer.where(aoi_mask)
ndvi_spring   = ndvi_spring.where(aoi_mask)
ndvi_summer   = ndvi_summer.where(aoi_mask)

# Optional county-level mean series for reporting
county_kndvi_spring = {k: kndvi_spring.where(m).mean(("lat", "lon")) for k, m in county_masks.items()}

Step 7: STL decomposition and residual extraction
Explanation. Equation 1 of your methodology decomposes each pixel's monthly kNDVI series into an additive trend, a seasonal cycle and a residual (Cleveland et al., 1990):
\mathrm{KNDVI}_t = \mathrm{Trend}_t + \mathrm{Season}_t + \mathrm{Residual}_t
STL needs a regular series without gaps, so it is run on the monthly composite (period = 12), not on the seasonal means; with one value per year there is no cycle to separate. Your methodology does not say how to treat missing months. The code therefore does not fill gaps by default: a pixel with any missing month returns NaN (MAX_MISSING_FRAC = 0.0). If this removes too many pixels, raise the threshold; pixels under the threshold are then gap-filled by linear interpolation, which is my choice and should be stated in your methods if used. The residual is finally averaged over spring and summer, the quantity your methodology uses to characterise the vegetation response

In [ ]:
MAX_MISSING_FRAC = 0.0     # your decision: share of missing months tolerated per pixel
STL_ROBUST       = False   # the methodology names plain STL; True gives the robust variant

def stl_residual_1d(y: np.ndarray) -> np.ndarray:
    """Residual of an additive STL fit; all-NaN output if the series fails the gap rule."""
    miss = np.isnan(y)
    if miss.all() or miss.mean() > MAX_MISSING_FRAC:
        return np.full_like(y, np.nan)
    if miss.any():                                   # only reachable if MAX_MISSING_FRAC > 0
        y = pd.Series(y).interpolate(limit_direction="both").to_numpy()
    res = STL(y, period=STL_PERIOD, robust=STL_ROBUST).fit()
    # sanity check of Equation 1: components must add back to the series
    assert np.allclose(res.trend + res.seasonal + res.resid, y, atol=1e-6)
    return np.asarray(res.resid, dtype="float32")

km = kndvi_monthly.transpose("time", "lat", "lon")
resid = xr.full_like(km, np.nan, dtype="float32")
for i, j in zip(*np.where(aoi_mask.values)):          # only pixels inside the AOI
    resid[:, i, j] = stl_residual_1d(km[:, i, j].values.astype("float64"))
resid.name = "kndvi_residual"

# Seasonal means of the residual, per year
resid_spring = seasonal_mean(resid, SPRING)
resid_summer = seasonal_mean(resid, SUMMER)

# Save outputs
xr.Dataset({
    "kndvi_spring": kndvi_spring, "kndvi_summer": kndvi_summer,
    "ndvi_spring":  ndvi_spring,  "ndvi_summer":  ndvi_summer,
    "resid_spring": resid_spring, "resid_summer": resid_summer,
}).to_netcdf(OUT_DIR / "taiwan_ndvi_kndvi_seasonal.nc")
kndvi_monthly.to_netcdf(OUT_DIR / "taiwan_kndvi_monthly_mvc.nc")

Output: seasonal NDVI, kNDVI and STL-residual rasters for 1985–2025, in outputs/. The loop runs on roughly a thousand pixels, which should take minutes, not hours; I have not timed it